# Última partida — dados completos (API-Football v3)

Este notebook resolve a **última partida** a analisar e pede **todos os sub-recursos** úteis para esse `fixture`: `fixtures` (por `ids`), **eventos**, **onzes**, **estatísticas de equipa**, **estatísticas de jogadores**, **odds**, **confrontos diretos** (`headtohead`) e **previsões**.

Os outputs privilegiam o **JSON completo** (`indent=2`, `ensure_ascii=False`) em cada pedido, para máxima informação.

**Documentação:** [API-Football v3](https://www.api-football.com/documentation-v3).

**Estratégia**

1. Se `USE_LAST` for `True`, tenta `GET /fixtures` com **`last`** (e opcionalmente **`team`**). Em muitos **planos Free**, a API devolve erro de plano para `last` — nesse caso passa ao passo 2.
2. **Fallback:** `GET /fixtures` com **`league`**, **`season`** e **`status`** (ex. jogos terminados), ordenação local por `fixture.date` (mais recente primeiro) e escolha de `LAST` jogo(s). O output do passo 2 **não** repete centenas de jogos: mostra metadados da resposta a granel + **só o(s) jogo(s) escolhido(s)** em JSON completo.

**Configuração:** célula seguinte (`USE_LAST`, `TEAM_ID`, `LAST`, `LEAGUE_ID`, `SEASON`, `STATUS`).

In [6]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

# Pasta do provedor: raiz do repo, `api-football/` ou `api-football/notebooks/` (cwd do Jupyter)
# Se o cwd do processo apontar para uma pasta já removida/renomeada, `Path.cwd()` levanta FileNotFoundError.
def _resolve_notebook_base() -> Path:
    try:
        return Path.cwd().resolve()
    except FileNotFoundError:
        pass
    # Depois de cwd inválido: preferir IPython/Jupyter antes de `PWD`.
    # `os.path.isdir(os.environ["PWD"])` pode bloquear muito tempo se `PWD` for rede/mount lento.
    try:
        from IPython import get_ipython

        sh = get_ipython()
        sd = getattr(sh, "starting_dir", None) if sh else None
        if sd and os.path.isdir(sd):
            return Path(sd).resolve()
    except Exception:
        pass
    for key in ("JUPYTER_SERVER_ROOT", "PWD"):
        raw = os.environ.get(key)
        if raw and os.path.isdir(raw):
            return Path(raw).resolve()
    raise RuntimeError(
        "O diretório de trabalho do kernel já não existe no disco (cwd inválido). "
        "Reinicia o kernel do Jupyter e abre o notebook a partir da pasta do repositório."
    ) from None


_nb_dir = _resolve_notebook_base()

def _find_api_football_provider(base: Path) -> Path:
    for c in (
        base / "api-football",
        base.parent if base.name == "notebooks" else None,
        base,
    ):
        if c is None:
            continue
        if (c / "authentication.py").is_file() and (c / "env_loader.py").is_file():
            return c.resolve()
    raise RuntimeError(
        "Não encontrei api-football/ (authentication.py). "
        f"cwd={base!s}. Usa cwd na raiz do repositório ou em api-football/notebooks."
    )


_provider = _find_api_football_provider(_nb_dir)
if str(_provider) not in sys.path:
    sys.path.insert(0, str(_provider))

# Carregar `.env` antes de ler `os.environ` (evita depender só do `load_local_env` dentro de `get_json`)
from env_loader import load_local_env

load_local_env()

BASE = os.environ.get("API_FOOTBALL_BASE_URL", "").rstrip("/")
LEAGUE_WORLD_CUP = 1
SEASON = os.environ.get("SEASON", "2026")

print(f"BASE: {BASE}")
print(f"LEAGUE_WORLD_CUP: {LEAGUE_WORLD_CUP}")
print(f"SEASON: {SEASON}")

BASE: https://v3.football.api-sports.io
LEAGUE_WORLD_CUP: 1
SEASON: 2022


In [2]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


# --- Parâmetro `last` (pode exigir plano pago) ---
USE_LAST: bool = True
TEAM_ID: int | None = None  # ex.: 33; só usado com `last`
LAST: int = 1  # quantos jogos: último(s) do feed `last` ou, no fallback, quantos trazer após ordenar por data

# --- Fallback plano Free: liga + temporada + estado ---
# Premier League 39 / temporada 2024 — ajuste conforme a tua subscrição e objectivo
LEAGUE_ID = 1
SEASON = 2022
# Estados curtos da API (ex.: FT terminado; pode usar "FT-AET-PEN" conforme doc)
STATUS = "FT"

print("BASE:", BASE or "(em falta)")
print("USE_LAST:", USE_LAST, "| TEAM_ID:", TEAM_ID, "| LAST:", LAST)
#print("Fallback league/season/status:", LEAGUE_ID, SEASON, repr(STATUS))

BASE: https://v3.football.api-sports.io
USE_LAST: True | TEAM_ID: None | LAST: 1


In [3]:
from authentication import get_json, verify_api_key

if not BASE:
    raise RuntimeError(
        "Defina API_FOOTBALL_BASE_URL em api-football/.env (ex.: https://v3.football.api-sports.io)"
    )


def dump(label: str, status: int, data: dict) -> None:
    print("\n" + "=" * 72)
    print(label)
    print("HTTP:", status)
    print(json.dumps(data, indent=2, ensure_ascii=False))


def fixture_id_from_row(row: dict) -> int | None:
    fix = row.get("fixture") or {}
    fid = fix.get("id")
    return int(fid) if fid is not None else None


check = verify_api_key()
print("Chave válida:", check.ok, "| HTTP:", check.http_status)
if not check.ok:
    dump("Resposta /timezone", check.http_status, check.data)
    raise RuntimeError("Chave inválida ou erros na API; corrija .env antes de continuar.")

Chave válida: True | HTTP: 200


## 1. Escolher a «última» partida (`last` ou fallback liga/temporada)

O objectivo é obter **um** `fixture.id` (o primeiro de `LAST` jogos). O output desta célula documenta **como** foi escolhido e inclui JSON completo do(s) jogo(s) seleccionado(s) (no fallback, não imprime a lista completa a granel).

In [4]:
MATCH_SOURCE = ""
rows: list = []
fixtures_payload: dict = {}

if USE_LAST:
    parts = [f"last={int(LAST)}"]
    if TEAM_ID is not None:
        parts.append(f"team={int(TEAM_ID)}")
    q_last = "&".join(parts)
    st, fixtures_payload = get_json(f"fixtures?{q_last}")
    rows = list(fixtures_payload.get("response") or [])
    errs = fixtures_payload.get("errors")
    if rows and not errs:
        MATCH_SOURCE = f"GET /fixtures?{q_last}"
        dump(f"Resolução da partida via {MATCH_SOURCE}", st, fixtures_payload)
    else:
        print(
            "\n(i) `last` sem dados ou com erros (ex. plano Free). "
            "Mensagem da API:",
            json.dumps(errs or {}, ensure_ascii=False),
        )

if not rows:
    q_fb = f"league={LEAGUE_ID}&season={SEASON}&status={STATUS}"
    st, bulk = get_json(f"fixtures?{q_fb}")
    all_rows = list(bulk.get("response") or [])
    if not all_rows:
        dump(f"Fallback GET /fixtures?{q_fb}", st, bulk)
        raise RuntimeError("Fallback sem jogos: ajuste LEAGUE_ID, SEASON ou STATUS.")
    all_rows.sort(
        key=lambda r: (r.get("fixture") or {}).get("date") or "",
        reverse=True,
    )
    n = max(int(LAST), 1)
    chosen = all_rows[:n]
    MATCH_SOURCE = f"fallback ordenado por data ← GET /fixtures?{q_fb}"
    fixtures_payload = {
        "get": bulk.get("get"),
        "parameters": bulk.get("parameters"),
        "errors": bulk.get("errors"),
        "paging": bulk.get("paging"),
        "results_bulk": bulk.get("results"),
        "_note": (
            "Resposta a granel truncada no output: só `response_selected` com JSON completo "
            f"({len(chosen)} jogo(s))."
        ),
        "response_selected": chosen,
        "results": len(chosen),
    }
    rows = chosen
    dump(f"Resolução da partida via {MATCH_SOURCE}", st, fixtures_payload)

TARGET_ID = fixture_id_from_row(rows[0])
if TARGET_ID is None:
    raise RuntimeError("Sem fixture.id no primeiro item seleccionado.")

print("\n--- Resumo ---")
print("Origem:", MATCH_SOURCE)
print("fixture_id (sub-endpoints):", TARGET_ID)


(i) `last` sem dados ou com erros (ex. plano Free). Mensagem da API: {"plan": "Free plans do not have access to the Last parameter."}

Resolução da partida via fallback ordenado por data ← GET /fixtures?league=1&season=2022&status=FT
HTTP: 200
{
  "get": "fixtures",
  "parameters": {
    "league": "1",
    "season": "2022",
    "status": "FT"
  },
  "errors": [],
  "paging": {
    "current": 1,
    "total": 1
  },
  "results_bulk": 59,
  "_note": "Resposta a granel truncada no output: só `response_selected` com JSON completo (1 jogo(s)).",
  "response_selected": [
    {
      "fixture": {
        "id": 979138,
        "referee": "Abdulrahman Al Jassim",
        "timezone": "UTC",
        "date": "2022-12-17T15:00:00+00:00",
        "timestamp": 1671289200,
        "periods": {
          "first": 1671289200,
          "second": 1671292800
        },
        "venue": {
          "id": 22430,
          "name": "Khalifa International Stadium",
          "city": "Ar-Rayyan"
        },
    

## 2. Detalhe adicional do mesmo jogo (`ids`)

Repete o enquadramento oficial do recurso `fixtures` filtrado por ID (útil para confirmar parâmetros e paging).

In [5]:
status, by_id = get_json(f"fixtures?ids={TARGET_ID}")
dump(f"GET /fixtures?ids={TARGET_ID}", status, by_id)


GET /fixtures?ids=979138
HTTP: 200
{
  "get": "fixtures",
  "parameters": {
    "ids": "979138"
  },
  "errors": {
    "plan": "Free plans do not have access to the Ids parameter."
  },
  "results": 0,
  "paging": {
    "current": 1,
    "total": 1
  },
  "response": []
}


## 3. Eventos, onzes, estatísticas, jogadores, confrontos diretos, odds e previsões

Cada chamada imprime o **JSON completo**. `headtohead` pode devolver vários jogos (histórico entre as duas equipas). `odds` pode vir vazio consoante o plano, a liga ou o mercado.

In [7]:
home_i = ((rows[0].get("teams") or {}).get("home") or {}).get("id")
away_i = ((rows[0].get("teams") or {}).get("away") or {}).get("id")

endpoints: list[tuple[str, str]] = [
    ("GET /fixtures/events", f"fixtures/events?fixture={TARGET_ID}"),
    ("GET /fixtures/lineups", f"fixtures/lineups?fixture={TARGET_ID}"),
    ("GET /fixtures/statistics", f"fixtures/statistics?fixture={TARGET_ID}"),
    ("GET /fixtures/players", f"fixtures/players?fixture={TARGET_ID}"),
    ("GET /odds", f"odds?fixture={TARGET_ID}"),
    ("GET /predictions", f"predictions?fixture={TARGET_ID}"),
]
if home_i and away_i:
    endpoints.insert(
        4,
        (
            "GET /fixtures/headtohead",
            f"fixtures/headtohead?h2h={int(home_i)}-{int(away_i)}",
        ),
    )

for label, path in endpoints:
    st, body = get_json(path)
    dump(label, st, body)


GET /fixtures/events
HTTP: 200
{
  "get": "fixtures/events",
  "parameters": {
    "fixture": "979138"
  },
  "errors": [],
  "results": 14,
  "paging": {
    "current": 1,
    "total": 1
  },
  "response": [
    {
      "time": {
        "elapsed": 7,
        "extra": null
      },
      "team": {
        "id": 3,
        "name": "Croatia",
        "logo": "https://media.api-sports.io/football/teams/3.png"
      },
      "player": {
        "id": 129033,
        "name": "J. Gvardiol"
      },
      "assist": {
        "id": 207,
        "name": "I. Perišić"
      },
      "type": "Goal",
      "detail": "Normal Goal",
      "comments": null
    },
    {
      "time": {
        "elapsed": 9,
        "extra": null
      },
      "team": {
        "id": 31,
        "name": "Morocco",
        "logo": "https://media.api-sports.io/football/teams/31.png"
      },
      "player": {
        "id": 36540,
        "name": "A. Dari"
      },
      "assist": {
        "id": null,
        "name": n

## 4. Resumo legível (além do JSON completo acima)

Extração mínima para leitura rápida; todo o detalhe continua nas células anteriores.

In [ ]:
row = ((by_id.get("response") or rows) or [{}])[0]
fx = row.get("fixture") or {}
lg = row.get("league") or {}
teams = row.get("teams") or {}
home = (teams.get("home") or {}).get("name")
away = (teams.get("away") or {}).get("name")
goals = row.get("goals") or {}
st = (fx.get("status") or {}).get("long")

print("Fixture ID:", fx.get("id"))
print("Data (API):", fx.get("date"), "|", fx.get("timezone"))
print("Estado:", st)
print("Competição:", lg.get("name"), "| temporada:", lg.get("season"), "| ronda:", lg.get("round"))
print("Casa:", home, "| Fora:", away)
print("Resultado (golos):", goals.get("home"), "-", goals.get("away"))
print("Árbitro:", (fx.get("referee")))
print("Estádio:", (fx.get("venue") or {}).get("name"), "|", (fx.get("venue") or {}).get("city"))